
# 04_evaluate_cleaned_200_value_level

对 `cleaned_test_set.json` 做 **value-level** 评估，专门适配这套 200 条测试集的特点：

1. **没有 span offset** → 不做 span-level F1，只做 **value-level matching**
2. **测试集有 79 种 PII**，但当前 student 只训练了 **16 种** → 评估时只看这 16 类
3. **GT 是 value-only schema**，模型输出是 tagged-text / span schema → 通过 `parse_annotated()` 桥接
4. **同一 value 可能在原文出现多次**，GT 只记一次 → 采用 **集合匹配**
5. **按 difficulty 分组** → `EASY / MEDIUM / HARD / EXPERT / TRAP`

> 说明：  
> 这个 notebook **不会**给出 span-level exact F1，因为这份 200 条测试集没有 offsets。  
> 它给的是：**value-level Precision / Recall / F1**、每类 breakdown、每个 difficulty breakdown、以及 TRAP 样本误报情况。


In [1]:

import json
import os
import re
import time
import unicodedata
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from redaction_utils import parse_annotated


/home/admin/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 配置

In [2]:

# ===== 路径 =====
BASE_MODEL   = "../../model/Qwen3.5-9B-Base"
ADAPTER_DIR  = "../outputs/qwen3_5_9b_base_lora_tagged_28_fastretry"
USE_MERGED   = False
MERGED_DIR   = "../outputs/qwen3_5_9b_base_lora_tagged_28_fastretry_merged"

# 建议把 cleaned_test_set.json 和 notebook 放在同一目录；否则改这里
TEST_JSON    = "../data/raw/cleaned_test_set.json"

# 可选：如果有 01 生成的 meta.json，会优先复用里面的 target_labels
META_PATH    = "../data/processed/meta.json"

TARGET_LABELS_PATH = os.path.join(ADAPTER_DIR, "target_labels.json")

REPORT_DIR   = "./eval_cleaned_200_report_9b"
os.makedirs(REPORT_DIR, exist_ok=True)

# ===== 推理参数 =====
BATCH_SIZE       = 8
MAX_INPUT_LEN    = 1536
MAX_NEW_TOKENS   = 1500
MAX_EXAMPLES     = None     # None = 跑完整 200 条；改成 20/50 可先试跑
SAVE_RAW_JSONL   = True

# ===== 只评估本次 adapter 实际训练过的类型 =====
if os.path.exists(TARGET_LABELS_PATH):
    with open(TARGET_LABELS_PATH, "r", encoding="utf-8") as f:
        TRAINED_TYPES = json.load(f)
    print(f"已从 adapter 读取 target_labels: {TARGET_LABELS_PATH}")
elif os.path.exists(META_PATH):
    with open(META_PATH, "r", encoding="utf-8") as f:
        TRAINED_TYPES = json.load(f)["target_labels"]
    print(f"已从 meta.json 读取 target_labels: {META_PATH}")
else:
    raise FileNotFoundError("找不到 target_labels.json 或 meta.json，无法确认本次评估标签集合")
TRAINED_SET = set(TRAINED_TYPES)

# GT 中哪些类型允许一个字段里出现多个逗号分隔的 value
# 只对这些类型做 split；其余类型保留原值（避免把 ADDRESS 拆坏）
MULTI_VALUE_TYPES = {
    "PERSON",
    "DATE_OF_BIRTH",
    "EMAIL_ADDRESS",
    "AU_PHONE",
    "AU_DRIVERS_LICENCE",
    "WORK_EMAIL",
    "WORK_PHONE",
}

# 可选：如果你想更严格，设为 False；如果想对大小写更宽松，保持 True
CASE_INSENSITIVE = True

assert os.path.exists(TEST_JSON), f"找不到测试集: {TEST_JSON}"
print("TEST_JSON   =", TEST_JSON)
print("ADAPTER_DIR =", ADAPTER_DIR)
print("USE_MERGED  =", USE_MERGED)
print("BATCH_SIZE  =", BATCH_SIZE)
print("只评估本次训练过的类型数 =", len(TRAINED_TYPES))
print("TRAINED_TYPES =", TRAINED_TYPES)


已从 adapter 读取 target_labels: ../outputs/qwen3_5_9b_base_lora_tagged_28_fastretry/target_labels.json
TEST_JSON   = ../data/raw/cleaned_test_set.json
ADAPTER_DIR = ../outputs/qwen3_5_9b_base_lora_tagged_28_fastretry
USE_MERGED  = False
BATCH_SIZE  = 8
只评估本次训练过的类型数 = 27
TRAINED_TYPES = ['PERSON', 'ADDRESS', 'DATE_OF_BIRTH', 'EMAIL_ADDRESS', 'AU_PHONE', 'AU_TFN', 'AU_PASSPORT', 'AU_DRIVERS_LICENCE', 'STUDENT_ID', 'MEDICARE_NUMBER', 'AU_BANK_ACCOUNT', 'BSB', 'PAYMENT_CARD_NUMBER', 'IP_ADDRESS', 'VEHICLE_REGO', 'SALARY', 'WORK_EMAIL', 'WORK_PHONE', 'EMPLOYEE_NUMBER', 'PERSONNEL_NUMBER', 'MEDICARE_EXPIRY', 'PASSPORT_EXPIRY', 'UAC_ID', 'USI', 'CENTRELINK_REFERENCE_NUMBER', 'CREDIT_CARD_EXPIRY', 'CREDIT_CARD_CVV']


## 2. 加载测试集，并统计其中有多少 GT 落在 16 个训练类型内

In [3]:

with open(TEST_JSON, "r", encoding="utf-8") as f:
    test_data = json.load(f)

if MAX_EXAMPLES is not None:
    test_data = test_data[:MAX_EXAMPLES]

print(f"测试样本数: {len(test_data)}")

difficulty_counter = Counter()
supported_gt_counter = Counter()
unsupported_gt_counter = Counter()
examples_with_supported_gt = 0
trap_count = 0

for ex in test_data:
    difficulty = ex.get("difficulty", "UNKNOWN")
    difficulty_counter[difficulty] += 1

    gt = ex.get("ground_truth_entities", {}) or {}
    has_supported = False
    if len(gt) == 0:
        trap_count += 1

    for k, v in gt.items():
        if k in TRAINED_SET:
            supported_gt_counter[k] += 1
            has_supported = True
        else:
            unsupported_gt_counter[k] += 1

    if has_supported:
        examples_with_supported_gt += 1

print("difficulty 分布:")
for k, v in difficulty_counter.items():
    print(f"  {k:<8s} {v}")

print(f"\n包含至少 1 个训练过 GT 类型的样本数: {examples_with_supported_gt}/{len(test_data)}")
print(f"TRAP 样本数: {trap_count}")

print(f"\n训练过的 {len(TRAINED_TYPES)} 类在这 200 条里的出现次数:")
for t in TRAINED_TYPES:
    print(f"  {t:<22s} {supported_gt_counter[t]}")

print("\n未训练类型（前 20）:")
for t, c in unsupported_gt_counter.most_common(20):
    print(f"  {t:<30s} {c}")


测试样本数: 200
difficulty 分布:
  EXPERT   50
  MEDIUM   50
  HARD     60
  TRAP     20
  EASY     20

包含至少 1 个训练过 GT 类型的样本数: 180/200
TRAP 样本数: 20

训练过的 27 类在这 200 条里的出现次数:
  PERSON                 180
  ADDRESS                118
  DATE_OF_BIRTH          108
  EMAIL_ADDRESS          78
  AU_PHONE               59
  AU_TFN                 61
  AU_PASSPORT            30
  AU_DRIVERS_LICENCE     14
  STUDENT_ID             48
  MEDICARE_NUMBER        12
  AU_BANK_ACCOUNT        12
  BSB                    12
  PAYMENT_CARD_NUMBER    8
  IP_ADDRESS             6
  VEHICLE_REGO           3
  SALARY                 13
  WORK_EMAIL             9
  WORK_PHONE             6
  EMPLOYEE_NUMBER        6
  PERSONNEL_NUMBER       3
  MEDICARE_EXPIRY        12
  PASSPORT_EXPIRY        8
  UAC_ID                 10
  USI                    10
  CENTRELINK_REFERENCE_NUMBER 10
  CREDIT_CARD_EXPIRY     8
  CREDIT_CARD_CVV        8

未训练类型（前 20）:
  WAM_SCORE                      17
  SEXUAL_ORIENTATION         

## 3. system prompt（优先复用训练时 meta.json）

In [4]:

SYSTEM_PROMPT = (
    "You are a PII annotator for Australian context.\n"
    "Return the SAME text with supported PII wrapped as <pii type=\"TYPE\">VALUE</pii>.\n"
    "Preserve every character exactly. Do not paraphrase, summarize, or explain.\n"
    "Wrap every occurrence of supported PII. Do not deduplicate.\n"
    "If no supported PII is present, return the input unchanged.\n"
    "Supported types:\n- " + "\n- ".join(TRAINED_TYPES)
)
print("使用与新版 SFT 对齐的 SYSTEM_PROMPT")


使用与新版 SFT 对齐的 SYSTEM_PROMPT


## 4. 加载模型

In [5]:

if USE_MERGED:
    print(f"加载 merged 模型: {MERGED_DIR}")
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
else:
    print(f"加载 base 模型: {BASE_MODEL}")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    print(f"挂载 adapter: {ADAPTER_DIR}")
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True, use_fast=False)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

print(f"padding_side = {tokenizer.padding_side}")
print(f"pad_token    = {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"eos_token    = {tokenizer.eos_token} (id={tokenizer.eos_token_id})")
if torch.cuda.is_available():
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


加载 base 模型: ../../model/Qwen3.5-9B-Base


Loading weights: 100%|████████████████████████████████████████████████████████████████| 427/427 [00:53<00:00,  7.95it/s]


挂载 adapter: ../outputs/qwen3_5_9b_base_lora_tagged_28_fastretry
padding_side = left
pad_token    = <|endoftext|> (id=248044)
eos_token    = <|endoftext|> (id=248044)
GPU allocated: 16.90 GB


## 5. 评估辅助函数

In [6]:

_THINK = re.compile(r"<think>.*?</think>\s*", re.DOTALL)

def normalize_value(x: str) -> str:
    x = unicodedata.normalize("NFKC", x)
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("–", "-").replace("—", "-")
    if CASE_INSENSITIVE:
        x = x.lower()
    return x

def split_gt_value(label: str, value: Any) -> List[str]:
    # GT 是 value-only schema。
    # - 普通类型: 保留一个原值
    # - 少数 multi-value 类型: 用逗号拆成多个值
    if value is None:
        return []
    if isinstance(value, list):
        out = []
        for v in value:
            out.extend(split_gt_value(label, v))
        return out

    s = str(value).strip()
    if not s:
        return []

    if label in MULTI_VALUE_TYPES:
        parts = [p.strip() for p in re.split(r"\s*,\s*", s) if p.strip()]
        return parts

    return [s]

def gt_to_pair_set(ex: Dict[str, Any]) -> set:
    gt = ex.get("ground_truth_entities", {}) or {}
    pairs = set()
    for label, value in gt.items():
        if label not in TRAINED_SET:
            continue
        for v in split_gt_value(label, value):
            pairs.add((label, normalize_value(v)))
    return pairs

def pred_to_pair_set(pred_spans: List[Dict[str, Any]]) -> set:
    pairs = set()
    for s in pred_spans:
        label = s.get("type")
        value = s.get("value")
        if label not in TRAINED_SET:
            continue
        if value is None:
            continue
        pairs.add((label, normalize_value(str(value))))
    return pairs

def build_prompt(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

@torch.no_grad()
def infer_batch(texts: List[str]) -> List[str]:
    prompts = [build_prompt(t) for t in texts]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LEN,
    ).to(model.device)

    out_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    outputs = []
    seq_len = inputs.input_ids.shape[1]
    for i in range(len(prompts)):
        gen = out_ids[i][seq_len:]
        outputs.append(tokenizer.decode(gen, skip_special_tokens=True).strip())
    return outputs

def parse_prediction(text: str, raw_output: str) -> Dict[str, Any]:
    cleaned = _THINK.sub("", raw_output).strip()
    try:
        plain, spans = parse_annotated(cleaned, strict=False)
        return {
            "cleaned_output": cleaned,
            "pred_spans": spans,
            "parse_ok": True,
            "round_trip_ok": (plain == text),
            "plain_text": plain,
            "parse_error": None,
        }
    except Exception as e:
        return {
            "cleaned_output": cleaned,
            "pred_spans": [],
            "parse_ok": False,
            "round_trip_ok": False,
            "plain_text": None,
            "parse_error": repr(e),
        }

def prf(tp: int, fp: int, fn: int) -> Dict[str, float]:
    p = tp / (tp + fp) if tp + fp > 0 else 0.0
    r = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2*p*r/(p+r) if p+r > 0 else 0.0
    return {"precision": p, "recall": r, "f1": f1}

def sample_exact_match(gt_pairs: set, pred_pairs: set) -> bool:
    return gt_pairs == pred_pairs


## 6. 跑 200 条推理

In [7]:

predictions = []

t0 = time.time()
for i in range(0, len(test_data), BATCH_SIZE):
    batch = test_data[i:i+BATCH_SIZE]
    texts = [ex["input"]["text"] for ex in batch]
    raws = infer_batch(texts)

    for ex, raw in zip(batch, raws):
        parsed = parse_prediction(ex["input"]["text"], raw)
        gt_pairs = gt_to_pair_set(ex)
        pred_pairs = pred_to_pair_set(parsed["pred_spans"])

        rec = {
            "id": ex["id"],
            "difficulty": ex.get("difficulty", "UNKNOWN"),
            "text": ex["input"]["text"],
            "raw_output": raw,
            "cleaned_output": parsed["cleaned_output"],
            "parse_ok": parsed["parse_ok"],
            "round_trip_ok": parsed["round_trip_ok"],
            "parse_error": parsed["parse_error"],
            "pred_spans": parsed["pred_spans"],
            "gt_pairs": sorted(list(gt_pairs)),
            "pred_pairs": sorted(list(pred_pairs)),
            "tp_pairs": sorted(list(gt_pairs & pred_pairs)),
            "fp_pairs": sorted(list(pred_pairs - gt_pairs)),
            "fn_pairs": sorted(list(gt_pairs - pred_pairs)),
            "sample_exact_match": sample_exact_match(gt_pairs, pred_pairs),
        }
        predictions.append(rec)

    done = min(i + BATCH_SIZE, len(test_data))
    elapsed = time.time() - t0
    rate = done / max(elapsed, 0.01)
    eta = (len(test_data) - done) / max(rate, 0.01)
    print(f"{done}/{len(test_data)} | {rate:.2f} samples/s | ETA {eta/60:.1f} min")

elapsed = time.time() - t0
print(f"\n推理完成: {len(predictions)} 条, 总耗时 {elapsed/60:.1f} min, 平均 {len(predictions)/elapsed:.2f} samples/s")

if SAVE_RAW_JSONL:
    raw_path = os.path.join(REPORT_DIR, "cleaned_200_predictions.jsonl")
    with open(raw_path, "w", encoding="utf-8") as f:
        for rec in predictions:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print("已保存:", raw_path)


8/200 | 0.30 samples/s | ETA 10.7 min
16/200 | 0.24 samples/s | ETA 12.5 min
24/200 | 0.25 samples/s | ETA 11.5 min
32/200 | 0.22 samples/s | ETA 12.7 min
40/200 | 0.23 samples/s | ETA 11.6 min
48/200 | 0.25 samples/s | ETA 10.3 min
56/200 | 0.22 samples/s | ETA 10.7 min
64/200 | 0.23 samples/s | ETA 9.8 min
72/200 | 0.24 samples/s | ETA 9.0 min
80/200 | 0.24 samples/s | ETA 8.3 min
88/200 | 0.24 samples/s | ETA 7.6 min
96/200 | 0.24 samples/s | ETA 7.2 min
104/200 | 0.24 samples/s | ETA 6.6 min
112/200 | 0.23 samples/s | ETA 6.3 min
120/200 | 0.22 samples/s | ETA 5.9 min
128/200 | 0.22 samples/s | ETA 5.4 min
136/200 | 0.23 samples/s | ETA 4.7 min
144/200 | 0.23 samples/s | ETA 4.0 min
152/200 | 0.23 samples/s | ETA 3.4 min
160/200 | 0.24 samples/s | ETA 2.8 min
168/200 | 0.24 samples/s | ETA 2.2 min
176/200 | 0.24 samples/s | ETA 1.7 min
184/200 | 0.24 samples/s | ETA 1.1 min
192/200 | 0.24 samples/s | ETA 0.6 min
200/200 | 0.24 samples/s | ETA 0.0 min

推理完成: 200 条, 总耗时 13.9 min, 平均 

## 7. 总体结果

In [8]:

overall_tp = overall_fp = overall_fn = 0
parse_ok = 0
round_trip_ok = 0
sample_exact_ok = 0

for rec in predictions:
    parse_ok += int(rec["parse_ok"])
    round_trip_ok += int(rec["round_trip_ok"])
    sample_exact_ok += int(rec["sample_exact_match"])
    overall_tp += len(rec["tp_pairs"])
    overall_fp += len(rec["fp_pairs"])
    overall_fn += len(rec["fn_pairs"])

overall = prf(overall_tp, overall_fp, overall_fn)

print(f"=== Overall (value-level, only {len(TRAINED_TYPES)} trained types) ===")
print(f"样本数               : {len(predictions)}")
print(f"Parse OK             : {parse_ok}/{len(predictions)}  ({parse_ok/len(predictions):.2%})")
print(f"Round-trip OK        : {round_trip_ok}/{len(predictions)}  ({round_trip_ok/len(predictions):.2%})")
print(f"Sample exact-set acc : {sample_exact_ok}/{len(predictions)}  ({sample_exact_ok/len(predictions):.2%})")
print(f"TP / FP / FN         : {overall_tp} / {overall_fp} / {overall_fn}")
print(f"Precision            : {overall['precision']:.4f}")
print(f"Recall               : {overall['recall']:.4f}")
print(f"F1                   : {overall['f1']:.4f}")


=== Overall (value-level, only 27 trained types) ===
样本数               : 200
Parse OK             : 200/200  (100.00%)
Round-trip OK        : 189/200  (94.50%)
Sample exact-set acc : 138/200  (69.00%)
TP / FP / FN         : 798 / 65 / 91
Precision            : 0.9247
Recall               : 0.8976
F1                   : 0.9110


## 8. 按 difficulty 分组

In [9]:

difficulty_rows = []

for diff in ["EASY", "MEDIUM", "HARD", "EXPERT", "TRAP", "UNKNOWN"]:
    rows = [r for r in predictions if r["difficulty"] == diff]
    if not rows:
        continue

    tp = fp = fn = 0
    parse_n = rt_n = exact_n = 0
    for r in rows:
        parse_n += int(r["parse_ok"])
        rt_n += int(r["round_trip_ok"])
        exact_n += int(r["sample_exact_match"])
        tp += len(r["tp_pairs"])
        fp += len(r["fp_pairs"])
        fn += len(r["fn_pairs"])

    m = prf(tp, fp, fn)
    difficulty_rows.append({
        "difficulty": diff,
        "n": len(rows),
        "parse_ok_rate": parse_n / len(rows),
        "round_trip_rate": rt_n / len(rows),
        "sample_exact_acc": exact_n / len(rows),
        "tp": tp, "fp": fp, "fn": fn,
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
    })

difficulty_rows


[{'difficulty': 'EASY',
  'n': 20,
  'parse_ok_rate': 1.0,
  'round_trip_rate': 1.0,
  'sample_exact_acc': 1.0,
  'tp': 88,
  'fp': 0,
  'fn': 0,
  'precision': 1.0,
  'recall': 1.0,
  'f1': 1.0},
 {'difficulty': 'MEDIUM',
  'n': 50,
  'parse_ok_rate': 1.0,
  'round_trip_rate': 0.86,
  'sample_exact_acc': 0.6,
  'tp': 194,
  'fp': 20,
  'fn': 27,
  'precision': 0.9065420560747663,
  'recall': 0.8778280542986425,
  'f1': 0.8919540229885058},
 {'difficulty': 'HARD',
  'n': 60,
  'parse_ok_rate': 1.0,
  'round_trip_rate': 0.9833333333333333,
  'sample_exact_acc': 0.6,
  'tp': 220,
  'fp': 23,
  'fn': 35,
  'precision': 0.9053497942386831,
  'recall': 0.8627450980392157,
  'f1': 0.8835341365461848},
 {'difficulty': 'EXPERT',
  'n': 50,
  'parse_ok_rate': 1.0,
  'round_trip_rate': 0.94,
  'sample_exact_acc': 0.64,
  'tp': 296,
  'fp': 22,
  'fn': 29,
  'precision': 0.9308176100628931,
  'recall': 0.9107692307692308,
  'f1': 0.9206842923794712},
 {'difficulty': 'TRAP',
  'n': 20,
  'parse_ok

## 9. 按类型 breakdown

In [10]:

per_type_stats = {t: {"tp": 0, "fp": 0, "fn": 0} for t in TRAINED_TYPES}

for rec in predictions:
    gt_pairs = set(tuple(x) for x in rec["gt_pairs"])
    pred_pairs = set(tuple(x) for x in rec["pred_pairs"])

    gt_by_type = defaultdict(set)
    pred_by_type = defaultdict(set)

    for t, v in gt_pairs:
        gt_by_type[t].add(v)
    for t, v in pred_pairs:
        pred_by_type[t].add(v)

    for t in TRAINED_TYPES:
        g = gt_by_type[t]
        p = pred_by_type[t]
        per_type_stats[t]["tp"] += len(g & p)
        per_type_stats[t]["fp"] += len(p - g)
        per_type_stats[t]["fn"] += len(g - p)

per_type_rows = []
for t in TRAINED_TYPES:
    s = per_type_stats[t]
    m = prf(s["tp"], s["fp"], s["fn"])
    support = s["tp"] + s["fn"]
    pred_n = s["tp"] + s["fp"]
    per_type_rows.append({
        "type": t,
        "support": support,
        "predicted": pred_n,
        "tp": s["tp"],
        "fp": s["fp"],
        "fn": s["fn"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
    })

per_type_rows


[{'type': 'PERSON',
  'support': 185,
  'predicted': 175,
  'tp': 166,
  'fp': 9,
  'fn': 19,
  'precision': 0.9485714285714286,
  'recall': 0.8972972972972973,
  'f1': 0.9222222222222223},
 {'type': 'ADDRESS',
  'support': 118,
  'predicted': 111,
  'tp': 110,
  'fp': 1,
  'fn': 8,
  'precision': 0.990990990990991,
  'recall': 0.9322033898305084,
  'f1': 0.9606986899563319},
 {'type': 'DATE_OF_BIRTH',
  'support': 124,
  'predicted': 113,
  'tp': 95,
  'fp': 18,
  'fn': 29,
  'precision': 0.8407079646017699,
  'recall': 0.7661290322580645,
  'f1': 0.8016877637130801},
 {'type': 'EMAIL_ADDRESS',
  'support': 83,
  'predicted': 77,
  'tp': 77,
  'fp': 0,
  'fn': 6,
  'precision': 1.0,
  'recall': 0.927710843373494,
  'f1': 0.9625},
 {'type': 'AU_PHONE',
  'support': 64,
  'predicted': 67,
  'tp': 61,
  'fp': 6,
  'fn': 3,
  'precision': 0.9104477611940298,
  'recall': 0.953125,
  'f1': 0.931297709923664},
 {'type': 'AU_TFN',
  'support': 61,
  'predicted': 61,
  'tp': 58,
  'fp': 3,
  '

## 10. TRAP 样本误报统计

In [11]:

trap_rows = [r for r in predictions if r["difficulty"] == "TRAP"]
trap_with_fp = [r for r in trap_rows if len(r["pred_pairs"]) > 0]

print(f"TRAP 样本数: {len(trap_rows)}")
print(f"TRAP 中有误报的样本数: {len(trap_with_fp)}")
print(f"TRAP 样本误报率: {len(trap_with_fp) / max(1, len(trap_rows)):.2%}")

print("\n前 10 个 TRAP 误报样本:")
for r in trap_with_fp[:10]:
    print("=" * 90)
    print("ID        :", r["id"])
    print("pred_pairs:", r["pred_pairs"])
    print("text      :", r["text"][:500])


TRAP 样本数: 20
TRAP 中有误报的样本数: 0
TRAP 样本误报率: 0.00%

前 10 个 TRAP 误报样本:


## 11. 看前几个误报 / 漏报例子

In [12]:

false_positive_examples = [r for r in predictions if len(r["fp_pairs"]) > 0]
false_negative_examples = [r for r in predictions if len(r["fn_pairs"]) > 0]
roundtrip_fail_examples = [r for r in predictions if not r["round_trip_ok"]]

print(f"FP 样本数             : {len(false_positive_examples)}")
print(f"FN 样本数             : {len(false_negative_examples)}")
print(f"Round-trip fail 样本数: {len(roundtrip_fail_examples)}")

print("\n=== 前 5 个 FP 样本 ===")
for r in false_positive_examples[:5]:
    print("=" * 100)
    print("ID         :", r["id"], "| difficulty =", r["difficulty"])
    print("FP pairs    :", r["fp_pairs"])
    print("GT pairs    :", r["gt_pairs"])
    print("Pred pairs  :", r["pred_pairs"])
    print("Text        :", r["text"][:600])

print("\n=== 前 5 个 FN 样本 ===")
for r in false_negative_examples[:5]:
    print("=" * 100)
    print("ID         :", r["id"], "| difficulty =", r["difficulty"])
    print("FN pairs    :", r["fn_pairs"])
    print("GT pairs    :", r["gt_pairs"])
    print("Pred pairs  :", r["pred_pairs"])
    print("Text        :", r["text"][:600])

print("\n=== 前 5 个 Round-trip fail 样本 ===")
for r in roundtrip_fail_examples[:5]:
    print("=" * 100)
    print("ID         :", r["id"], "| difficulty =", r["difficulty"])
    print("parse_ok    :", r["parse_ok"], "| parse_error =", r["parse_error"])
    print("raw_output  :", r["raw_output"][:800])


FP 样本数             : 50
FN 样本数             : 60
Round-trip fail 样本数: 11

=== 前 5 个 FP 样本 ===
ID         : TEST-005 | difficulty = HARD
FP pairs    : [('AU_TFN', 'tfn: 832 109 111'), ('DATE_OF_BIRTH', 'dob: 25/06/1969'), ('PERSON', 'kowalski')]
GT pairs    : [('ADDRESS', '305 george ln, darwin nt 0810'), ('AU_TFN', '832 109 111'), ('DATE_OF_BIRTH', '25/06/1969'), ('PERSON', 'marco kowalski')]
Pred pairs  : [('ADDRESS', '305 george ln, darwin nt 0810'), ('AU_TFN', 'tfn: 832 109 111'), ('DATE_OF_BIRTH', 'dob: 25/06/1969'), ('PERSON', 'kowalski')]
Text        : Marco Kowalski commenced employment on 01/07/2024. DOB: 25/06/1969. Contract expires: 31/12/2025. TFN: 832 109 111. Address: 305 George Ln, Darwin NT 0810.
ID         : TEST-006 | difficulty = MEDIUM
FP pairs    : [('DATE_OF_BIRTH', 'dob: 24 may 1989')]
GT pairs    : [('AU_DRIVERS_LICENCE', 'nsw 35241543'), ('AU_DRIVERS_LICENCE', 'o8823757'), ('AU_DRIVERS_LICENCE', 'qld: 993073270'), ('DATE_OF_BIRTH', '24 may 1989'), ('PERSON', 'ibr

## 12. 保存汇总报告

In [13]:

summary = {
    "n_examples": len(predictions),
    "trained_type_count": len(TRAINED_TYPES),
    "trained_types": TRAINED_TYPES,
    "overall": {
        "parse_ok_rate": parse_ok / len(predictions),
        "round_trip_rate": round_trip_ok / len(predictions),
        "sample_exact_acc": sample_exact_ok / len(predictions),
        "tp": overall_tp,
        "fp": overall_fp,
        "fn": overall_fn,
        "precision": overall["precision"],
        "recall": overall["recall"],
        "f1": overall["f1"],
    },
    "difficulty_breakdown": difficulty_rows,
    "per_type_breakdown": per_type_rows,
    "trap": {
        "n_trap": len(trap_rows),
        "n_trap_with_fp": len(trap_with_fp),
        "trap_fp_rate": len(trap_with_fp) / max(1, len(trap_rows)),
    },
    "notes": [
        "This is value-level evaluation only.",
        f"Only the {len(TRAINED_TYPES)} trained types are evaluated.",
        "Unsupported GT types are ignored by design.",
        "GT has no offsets, so span-level F1 is intentionally not reported.",
    ],
}

summary_path = os.path.join(REPORT_DIR, "cleaned_200_value_level_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("已保存 summary:", summary_path)
print(json.dumps(summary["overall"], ensure_ascii=False, indent=2))


已保存 summary: ./eval_cleaned_200_report_9b/cleaned_200_value_level_summary.json
{
  "parse_ok_rate": 1.0,
  "round_trip_rate": 0.945,
  "sample_exact_acc": 0.69,
  "tp": 798,
  "fp": 65,
  "fn": 91,
  "precision": 0.9246813441483198,
  "recall": 0.8976377952755905,
  "f1": 0.910958904109589
}
